`conda activate scenicplus`

In [1]:
import os
import gzip
import subprocess
from concurrent.futures import ProcessPoolExecutor
import anndata as ad
import scanpy as sc

# Define the directory containing the fragment files and the output file name
input_directory = "/projects/0/einf2548/cruiz/dmg/public_data/Mannens2024/compiled_fragments/samples"
output_file = "/projects/0/einf2548/cruiz/dmg/public_data/Mannens2024/compiled_fragments/compiled_filtered_fragments.tsv"
num_threads = 96  # Adjust the number of threads as needed

# Load the AnnData object
atac = sc.read('/projects/0/einf2548/cruiz/dmg/notebooks/scATAC/data/Mannens2024/Pool_peaks.h5ad')

atac.obs_names = atac.obs_names + '-1'
# Extract barcodes from the AnnData object
atac_barcodes = set(atac.obs.index)
atac_barcodes

Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)/home/cruiz2/miniconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


{'10X242_1:TTCATCACAGAGATGC-1',
 '10X369_3:TAATTCCTCGCTACCT-1',
 '10X369_1:GTGTCCTTCGTTCTAT-1',
 '10X321_2:CTTGCTGTCTGTGTGA-1',
 '10X366_1:CAACGTAAGCAGTAGC-1',
 '10X407_2:TAAGTGCGTCCAGACC-1',
 '10X315_6:CAATCCCCACTAGGAG-1',
 '10X280_2:CTTACCGGTGCTCCGT-1',
 '10X325_6:GCGCCAATCCAATCCC-1',
 '10X346_1:CCATAGCCACCCACCT-1',
 '10X280_3:GTAGGCGAGGTGTTAC-1',
 '10X372_3:CGCAATAAGCTGAAAT-1',
 '10X346_2:TATTTGGAGCGGATTT-1',
 '10X279_2:GTCACCTTCCCGTATC-1',
 '10X406_7:TGGATTCAGCAGCTAT-1',
 '10X317_1:CTGGCAGGTGATAAGT-1',
 '10X406_6:GGAGTGAGTTCATTTG-1',
 '10X406_4:ATGACAACAGGATGGC-1',
 '10X340_4:TGACAACAGAAAGGGT-1',
 '10X372_4:GTTATTCTCACCACAA-1',
 '10X370_3:GACTATTCATCCGTAA-1',
 '10X313_1:AGCCTGGCATGGCCGT-1',
 '10X312_3:GAGGATGGTGTTAGAA-1',
 '10X250_1:GAAATGAGTCTGTTGA-1',
 '10X242_2:ATAGTCGCAGTCCTGG-1',
 '10X366_6:GCTTGCTCATGATCGT-1',
 '10X340_1:AACTTGGGTACGACCC-1',
 '10X369_2:CCCTGATCACGTTGTA-1',
 '10X280_3:AGGATATAGGACCAGG-1',
 '10X290_2:GGATTGCGTCATGCCC-1',
 '10X365_2:CTTTGTCCAATCCCTT-1',
 '10X340

In [2]:
del atac

In [3]:
import gc
gc.collect()

518

In [4]:
sorted_output_file = f"{output_file}.sorted"

# Create a function to process, modify barcodes, filter, and write to disk
def process_file(filename):
    full_path = os.path.join(input_directory, filename)
    prefix = filename.split("_fragments.tsv.gz")[0]
    temp_output = []

    with gzip.open(full_path, 'rt') as infile:
        for line in infile:
            parts = line.strip().split("\t")
            if len(parts) >= 4:
                # Modify the barcode with the prefix and colon
                barcode = parts[3]
                modified_barcode = f"{prefix}:{barcode}"
                
                # Replace the original barcode with the modified one
                parts[3] = modified_barcode
                
                # Only add lines where the modified barcode is in atac_barcodes
                if modified_barcode in atac_barcodes:
                    temp_output.append("\t".join(parts) + "\n")
    
    return temp_output

def concatenate_fragments(input_directory, output_file):
    with open(output_file, 'w') as outfile:
        with ProcessPoolExecutor() as executor:
            # Get all fragment files
            files = [f for f in os.listdir(input_directory) if f.endswith("fragments.tsv.gz")]
            
            # Process files in parallel
            results = executor.map(process_file, files)
            
            for result in results:
                outfile.writelines(result)

# Run the function to process, filter, and write fragments to disk
concatenate_fragments(input_directory, output_file)

KeyboardInterrupt: 

In [6]:
def sort_and_compress(output_file, sorted_output_file, num_threads):
    # Set the environment variable LC_ALL to C for sorting
    env = os.environ.copy()
    env['LC_ALL'] = 'C'

    # Sort the file using the Unix sort command with LC_ALL=C
    subprocess.run(["sort", "--parallel=96", "-k1,1", "-k2,2n", "-k3,3n", output_file, "-o", sorted_output_file], check=True, env=env)

    # Compress the sorted file using bgzip
    subprocess.run(["bgzip", "--threads", str(num_threads), sorted_output_file], check=True)

    # Index the bgzip-compressed file with tabix
    subprocess.run(["tabix", "--threads", str(num_threads), "-p", "bed", f"{sorted_output_file}.gz"], check=True)

# Sort, compress, and index the output file
sort_and_compress(output_file, sorted_output_file, num_threads)

print(f"Filtered, sorted, compressed, and indexed fragments file created: {sorted_output_file}.gz")

Filtered, sorted, compressed, and indexed fragments file created: /projects/0/einf2548/cruiz/dmg/public_data/Mannens2024/compiled_fragments/compiled_filtered_fragments.tsv.sorted.gz
